# Classwork: The Analyst's Toolkit

**Parker Simms**

After learning about different tools (or a toolkit-- Descriptive statistics, Correlation, Regression/Multiple regression, Chi-square test) of working with data in the previous Mods, in this acitivity you will apply your real skills on the data you have.  
  
  Suppose your company hands you `customers.csv` (40 customers: `age`, `income`, `spending`, `region`, `subscribed`) and a list of questions. You are the analyst. Answer each one with the right tool/technique you decide to apply.

Work in pairs. Write answers in the markdown cells provided.

## Setup
Import the libraries you'll need and load the data.

In [1]:
# import pandas (and numpy, scipy.stats, statsmodels.api as you need them)
# load customers.csv into a DataFrame
# take a first look (head, info)

import pandas as pd
from scipy import stats
import statsmodels.api as sm

df = pd.read_csv('customers.csv')

print(df.head(23))
df.info()

    customer_id  age  income  spending region subscribed
0             1   25    61.3      39.1   West         No
1             2   25    54.9      42.1  South        Yes
2             3   51   105.1      59.9  South         No
3             4   39     NaN      49.9  South         No
4             5   43    72.4      48.3   East         No
5             6   44    84.7      47.1   East         No
6             7   48    92.9      60.0   East        Yes
7             8   21    53.3      36.0  South         No
8             9   39    71.9      48.9   East        Yes
9            10   25    70.4      42.1   east        Yes
10           11   36    92.2      62.6   East        Yes
11           12   57   106.1      66.7   East        Yes
12           13   41    81.8      49.8  South        Yes
13           14   22    70.2      48.9   West         No
14           15   41    96.4      65.8   East        Yes
15           16   25    67.0      54.2   West        Yes
16           17   50   105.4   

## Part 0: Know your data, then clean it

This data is dirty on purpose. Find and fix **three** problems before you analyze anything.

In [2]:
# HINT 2: one category label is written in the wrong case -> standardize it
# HINT 3: one value is impossible for its column -> treat it as missing, then fill
# tip: check .isna().sum() and the unique values of the category column

# Dirt 3: Impossible Value
df.loc[df['income'] < 0, 'income'] = pd.NA   #note: had to change the order, since it was causing errors in the next cells

# Dirt 1: Missing Values
df['income'] = df['income'].fillna(df['income'].median())

# Dirt 2: Standardizing Label
df['region'] = df['region'].str.title()    



print(df.isna().sum())
print(df['region'].unique())



customer_id    0
age            0
income         0
spending       0
region         0
subscribed     0
dtype: int64
<StringArray>
['West', 'South', 'East']
Length: 3, dtype: str


**Explain each issue:** what you found and how you handled it.

- One of the first issues I had noticed is the NaN value in the 'income' category. To fix this, I used the df[].fillna logic to fill those values with the MEDIAN value of that column. 

- Another issue I noticed was when I loaded all 20 of the values in this set, and saw that for 'region', there is a row that is lowercase, as opposed to uppercase like every other entry. To fix this, I used str.title(), which would turn that lowercase east into a "titlecase" East.

- Customer 26 has an income of -50.0, which is an impossible value. To fix this, I put in a check to convert any values below zero into a NaN value.


## Part 1: Pick the tool for each question

For each question: (a) name the data types involved, (b) name the tool, (c) run it, (d) answer in one plain sentence.

### Q1. Do customers who earn more also spend more?

In [3]:
# data types: income = continuous, spending = continuous
# tool: correlation, since it measures how two numeric values move together

r, p = stats.pearsonr(df['income'], df['spending'])
print(r, p)


0.7172222861355749 1.9185294399175956e-07


*Your answer:*
Yes, I think higher income customers do spend more (looking at r = 0.72 and p < 0.001), since the numbers show a strong and positive correlation.



### Q2. Is a customer's region related to whether they subscribed?

In [4]:
# data types: region = categorical, subscribed = categorical
# tool: chi-square (since categorical)

table = pd.crosstab(df['region'], df['subscribed']) #chi-square needs the table this makes for its input/basis
print(table)

chi2, p, dof, expected = stats.chi2_contingency(table) # takes the prev table and does a chi-square calculation on it
print(chi2, p, dof)

subscribed  No  Yes
region             
East         5   16
South        6    3
West         9    1
13.161904761904761 0.0013865281628130974 2


*Your answer:*
I think that there is a visible correlation between region and subscription status, as 16/21 subscribers are from the East, and 9/10 in the West are NOT subscribers.

### Q3. Can we predict spending from income and age together? How much do they explain? (You can review your Assignment 4 for this)

In [5]:
# data types: predicting a continuous outcome (spending) from TWO predictors (income/age)
# tool: multiple regression

x = sm.add_constant(df[['income', 'age']])
  #add_constant adds a col of 1s so the model can est an intercept, and [['income', 'age']] selects the two predictor variable cols
model = sm.OLS(df['spending'], x).fit() 
  #sm.OLS sets up the model (ordinary least squares) which puts the outcome first and the predictors second, and fit() finds the best fit coefficients
print(model.summary)

<bound method RegressionResults.summary of <statsmodels.regression.linear_model.OLSResults object at 0x000001E0811D2DE0>>


*Your answer:*


### Q4. Which region has the most customers, and what is the average income overall?

In [8]:
print(df['region'].value_counts()) # counts rows per cat
print(df['income'].mean()) # takes avg for the col


region
East     21
West     10
South     9
Name: count, dtype: int64
86.46


*Your answer:*
- East has the most customers (21) and the avg is about 86.5

## Part 2: The reveal

Answer these two in a sentence each (no code needed).

**a)** Look back at your four tools (Correlation, Chi-square test, Multiple regression, & Descriptive statistic)). What decided which tool you used each time? (Hint: it was not the wording of the question.)

*Your answer:*
- The DATA TYPES decided the tool instead of the wording. Continuous = correlation, and categorical = chi square

**b)** In Q1 you found a relationship. Can you tell the company that higher income *causes* higher spending? Why or why not?

*Your answer:*
- No because correlation shows that income and spending are moving together, not that income directly leads to spending.

## Bonus

In Q2 you ran a chi-square test. Look at its **expected counts** (not the observed counts). Do they all meet the assumption you learned in the chi-square module? What would you tell the company about how much to trust that result?

In [9]:
# hint: chi2_contingency returns the expected table as its 4th value
# check: are all expected counts >= 5?

print(expected)
print((expected >= 5).all())


[[10.5 10.5]
 [ 4.5  4.5]
 [ 5.   5. ]]
False


*Your answer:*
Not all of the expected counts meet the assumption, since the pattern is strong but the p value is small. It's probably a real and justifiable result but I'm not so sure if it should be 100% trusted.


### Push your code to GitHub